# Phase 02 — Experimental schema and quality control

**Project:** Open Materials Data Pipeline for Hydrogel Bioink Design
**Source:** [Zenodo record 19602891](https://zenodo.org/records/19602891), version v0.0.2

**What this notebook does.** It turns the Phase 01 audit into a schema: what each row represents, which keys identify it, how each source header maps to a defined field and unit, and which quality-control rules apply. The schema is then tested on the real files: every worksheet, measurement point and source cell is represented with its original coordinates, every image-index record keeps its blanks as "unknown", and executable checks confirm nothing was lost, duplicated or silently changed. The percent-to-fraction strain rule is previewed but not applied; viscosity units are not converted.

| | |
|---|---|
| **Inputs** | The two source files in `data/raw/zenodo_19602891/`; the 11 Phase 01 tables and `audit_run_manifest.json` in `data/metadata/zenodo_source_audit/` |
| **Outputs** | 26 tables (CSV) and `schema_qc_run_manifest.json` in `data/metadata/schema_and_quality_control/`; `phase02_checks.csv` in the same folder |
| **Runtime** | About 15 seconds |
| **Requires** | Phase 01 run in this project folder (14 of 14 checks passed) |

**How to run.** Select the kernel built from `environment.yml`, then **Kernel → Restart Kernel and Run All Cells**. Section 12 reports every check as **PASS**, **WARN** or **FAIL**; a failure stops the notebook with the reason. Section 13 lists the findings with the status labels defined in Phase 00: **Verified**, **Preliminary**, **Assumption**, **Unresolved**.

**Scope.** Keys identify digital records (worksheets, rows, cells), not physical samples. No sample, formulation, printing-run or construct identifiers are created; those entities stay deferred until the source supports them.

## 1. Settings and helper functions

Paths, expected counts and helper functions. `require` stops the notebook when a structural condition fails. `record` adds a row to this phase's check table (section 12). The reference checksums identify the Phase 01 tables this schema was designed against, so a changed audit is detected before it is used.

`reviewed_inputs` is a historical register of the documents consulted when the schema was designed (file names as they were on the author's computer at the time). It is exported for provenance only; it is not read at run time.

In [1]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from importlib.metadata import version
import csv
import hashlib
import io
import json
import platform
import zipfile

import numpy as np
import openpyxl
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", None)

AUDIT_REL = Path("data/metadata/zenodo_source_audit")
RAW_REL = Path("data/raw/zenodo_19602891")
OUT_REL = Path("data/metadata/schema_and_quality_control")

PHASE01_REFERENCE_SHA256 = {   # the 11 Phase 01 tables this schema was designed and tested against
    "source_manifest.csv": "5fefea8090bbd598e12020f5b1be93eccde87a7a4b83faaa7836a5f54fa48be7",
    "archive_member_inventory.csv": "a82ea7580b4941869ca0c1b2698efd50e7e4feec57678cfd0497237d0eb50751",
    "workbook_structure_inventory.csv": "e546d31e4f0434a9e797ee3555ca71b25afb653162dbacfc793a3ee813f1ee9a",
    "source_header_inventory.csv": "8184bfff6a5d13b587044bfd0098f266c78542fb4aad30e6c1a1637d8f3b0ed7",
    "measurement_column_profile.csv": "120aa04a12e56993d3821dc82be5569af30fe074039f509023f3fd05c236dca8",
    "measurement_axis_profile.csv": "c0a30820727068b30ab3ca7d8a2aa49df353e5599982c2a5c07ddcf3c85bc40c",
    "image_index_column_profile.csv": "5885ff29927a70921059954aacacba7eba8ff33bcbad253337e99979a0459a60",
    "image_index_value_counts.csv": "e2fbe1568b7d0043c5750f6fbfa05fdb52fccd34afde73b1af978db879460666",
    "candidate_filename_correspondences.csv": "a3f95d08e9393b7e871d9a2103b85f2c0306bb46357d36b8ac03cd6496e4003e",
    "image_filename_expectations.csv": "6f0aff02ad2afb05ae39c53ff9da7313fbabd033f392253769d5d4acbb3ab759",
    "audit_issue_log.csv": "1b898b430040b1ea0e5cc76362923d3f7bc1d830e49d5eb60ef8223da4f6013e",
}
EXPECTED = {"worksheets": 92, "points": 4600, "cells": 39100, "image_records": 1031,
            "dictionary_fields": 37, "structural_checks": 43, "value_findings": 24,
            "nonpositive_cells": 20, "nonpositive_worksheets": 8, "exports": 26}
REFERENCE_EXPORT_SHA256 = {   # SHA-256 of each output table from the tested reference run
    "input_verification.csv": "25a7af10c26835e88d2d43314532b07ae53040fea38637d3444d8d03bc5da05c",
    "reviewed_inputs.csv": "61188081050fd080792c106f667800d136fe94b975c6c1250c8980d27da41f21",
    "literature_evidence_register.csv": "01be2f6d3cb0e59740b631da534e7bbefb7a8d5f4744c156b9711dcfaabf824f",
    "entity_schema.csv": "b9901b9b5f3f73a486973b9f9a064e61001d47c57579e40b352aa5d4c88d7eaa",
    "deferred_entities.csv": "d25b05900e44a959a7f762a66c3c918f4e20341f3a067ef72cf83489f4824572",
    "source_to_schema_mapping.csv": "9545f539a2ff545771d5c2a0e6f6946846800a79d755cebfbf39d34a80910750",
    "image_field_dictionary.csv": "dcf5b68b8c15cb209559d27729fd5a45fc36fcef24f1e1179607bb148c9a962b",
    "data_dictionary.csv": "4d3985143ca73b5c7ea242ae940320233a5cb66657c0ec68ae8e4e412a734d2e",
    "missing_value_rules.csv": "e0c235fafc123a0d853baf0733472c8328e5249bfe2e12c106c7249eb200ebf3",
    "worksheet_register.csv": "55cf6681264c510d30a9c027d03f5dd536d3cea934b579935fd272372f683995",
    "measurement_points.csv": "4f690ae7fddc9b4cca5294250859c2fafabb0927dcf0dfdad79a609e7f8ec78f",
    "measurement_values_source.csv": "2f495ed1545a61e88a5ef181a15e97ea67e4ba4b5ebebfb8c2029fc8f6e52687",
    "image_records_source.csv": "003a0a0ffb7f807646332aaafc14ab08b4f9d627e5b82bdceba440b23f1f5fc4",
    "image_missingness.csv": "d1dc372cd1b51c3a185b97db35e36dfbf277dd1d72f997c584ca14089a5e7ec2",
    "image_observed_categories.csv": "e2fbe1568b7d0043c5750f6fbfa05fdb52fccd34afde73b1af978db879460666",
    "quality_control_rules.csv": "31d70daddbe1ebcece094bbb4286cef8615e4fdda53b80fd9bccedb156024357",
    "validation_summary.csv": "4a438638c9151df082f2e03ef8817df07e7f502a6a5473b5ba51ba8e1b1b45a9",
    "value_qc_findings.csv": "469e1c663d1bf6d03c9a7f515fb50daf8f387331e678e265335e0dd38a67a9a1",
    "strain_conversion_preview.csv": "dab0b045756c37055fd7993f4c22eed5c832969f62a7ecfd27ffa5db25a5bfb1",
    "measurement_context_summary.csv": "9aa021f4e0b63a1b035063cf5f734a036bb21e3bdb2b4dcb3cb509a055a98c45",
    "context_review.csv": "27f95971c44a719d85345413b86ce890ca3515ab30281a4d1278970cde22df64",
    "phase1_issue_carry_forward.csv": "1b898b430040b1ea0e5cc76362923d3f7bc1d830e49d5eb60ef8223da4f6013e",
    "phase2_issue_decisions.csv": "b3ad54ad613940fdcb60e168c7eeb21c49c1df4dc50de012a5884f5ebce5a580",
    "additional_review_items.csv": "c26e2ceede6b2af554617c0e5132749c11dc55213fa612f9daca5291bd941406",
    "filename_candidates_unconfirmed.csv": "a3f95d08e9393b7e871d9a2103b85f2c0306bb46357d36b8ac03cd6496e4003e",
    "linkage_summary.csv": "3d36ab09fb6e12714d59f3028b6ef187e589e3b2cc35fd3786536d0b6f582510",
}

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

def is_blank(value):
    return value is None or (isinstance(value, str) and not value.strip())

phase_checks = []   # this phase's PASS/WARN/FAIL checks (section 12)

def record(check_id, description, passed, detail, warn_only=False):
    status = "PASS" if passed else ("WARN" if warn_only else "FAIL")
    phase_checks.append({"check_id": check_id, "description": description, "status": status, "detail": detail})
    print(f"[{status:4}] {check_id}: {detail}")
    if status == "FAIL":
        raise RuntimeError(f"{check_id} failed: {detail}")

environment = pd.DataFrame([{"component": "Python", "version": platform.python_version()}] +
                           [{"component": p, "version": version(p)} for p in ["pandas", "numpy", "openpyxl", "ipython"]])

reviewed_inputs = pd.DataFrame([{'reviewed_filename': '01_zenodo_source_audit(4).ipynb', 'sha256': 'e4de9e1602e54b71fa954c7480f12a233c9376d07d23208f70397df8590001bc', 'role': 'phase1_checkpoint'}, {'reviewed_filename': 'ALG-Ph_HA-Ph_rheology_data(3).zip', 'sha256': '8498a0a1c878bd1855d6767078f70b27e950b52b8b180201a46214063ed65ff4', 'role': 'data_or_audit'}, {'reviewed_filename': 'ChatGPT Prompt for Hydrogel Bioink(6).docx', 'sha256': 'a7de7e06fd5ffb3c9dee453739703b3b28fd4b6cb2509cd36b538a64c541c0a5', 'role': 'supporting_document'}, {'reviewed_filename': 'GENERA~1(1).DOC', 'sha256': '16bfe75b0ddf0be5029d384fed2f888ffdad9c0fe80e2803850d0b1cd2f98e81', 'role': 'supporting_document'}, {'reviewed_filename': 'Generative AI-guided in silico closed-loop optimisation of deposition morphology for 3D bioprinting applications(1).pdf', 'sha256': 'c3868eb63b4f4ce96be0b39ea4b058cc91ccf86d98697ff4ca673bfb560b6536', 'role': 'supporting_document'}, {'reviewed_filename': 'README_Phase1(1).txt', 'sha256': '9a4d8c75f5fa664bab50565e2d1472204cc00f9b86a57511a6fa6b4f875f74c3', 'role': 'supporting_document'}, {'reviewed_filename': 'archive_member_inventory(1).csv', 'sha256': 'a82ea7580b4941869ca0c1b2698efd50e7e4feec57678cfd0497237d0eb50751', 'role': 'data_or_audit'}, {'reviewed_filename': 'audit_issue_log(1).csv', 'sha256': '1b898b430040b1ea0e5cc76362923d3f7bc1d830e49d5eb60ef8223da4f6013e', 'role': 'data_or_audit'}, {'reviewed_filename': 'audit_run_manifest(1).json', 'sha256': 'a28b0638ef896d2c3ba04b65b8cd66f60a2d292f120b0eb28f91726876724193', 'role': 'data_or_audit'}, {'reviewed_filename': 'candidate_filename_correspondences(1).csv', 'sha256': 'a3f95d08e9393b7e871d9a2103b85f2c0306bb46357d36b8ac03cd6496e4003e', 'role': 'data_or_audit'}, {'reviewed_filename': 'image_filename_expectations(1).csv', 'sha256': '6f0aff02ad2afb05ae39c53ff9da7313fbabd033f392253769d5d4acbb3ab759', 'role': 'data_or_audit'}, {'reviewed_filename': 'image_index(3).csv', 'sha256': '6e51d222d5434e271e4757ea750b8b4b002df13f056c21a5037d0e0b33a7b2c5', 'role': 'data_or_audit'}, {'reviewed_filename': 'image_index_column_profile(1).csv', 'sha256': '5885ff29927a70921059954aacacba7eba8ff33bcbad253337e99979a0459a60', 'role': 'data_or_audit'}, {'reviewed_filename': 'image_index_value_counts(1).csv', 'sha256': '40b1542dbf9d024e5db0d39b903035c4c1ad70a791704a2ed7a882ae2515dfd9', 'role': 'data_or_audit'}, {'reviewed_filename': 'measurement_axis_profile(1).csv', 'sha256': 'c0a30820727068b30ab3ca7d8a2aa49df353e5599982c2a5c07ddcf3c85bc40c', 'role': 'data_or_audit'}, {'reviewed_filename': 'measurement_column_profile(1).csv', 'sha256': '120aa04a12e56993d3821dc82be5569af30fe074039f509023f3fd05c236dca8', 'role': 'data_or_audit'}, {'reviewed_filename': 'source_header_inventory(1).csv', 'sha256': '8184bfff6a5d13b587044bfd0098f266c78542fb4aad30e6c1a1637d8f3b0ed7', 'role': 'data_or_audit'}, {'reviewed_filename': 'source_manifest(2).csv', 'sha256': '5fefea8090bbd598e12020f5b1be93eccde87a7a4b83faaa7836a5f54fa48be7', 'role': 'data_or_audit'}, {'reviewed_filename': 'workbook_structure_inventory(2).csv', 'sha256': 'e546d31e4f0434a9e797ee3555ca71b25afb653162dbacfc793a3ee813f1ee9a', 'role': 'data_or_audit'}])
print("Settings loaded. Python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__)

Settings loaded. Python 3.12.16 | pandas 2.2.3 | numpy 2.5.3


## 2. Verify the Phase 01 handoff

The project folder is found the same way as in Phases 00 and 01. Every file listed in `audit_run_manifest.json` (the two source files and the 11 audit tables) must exist and match the SHA-256 recorded there. The 11 tables must also match the reference checksums in section 1, which confirms the schema is applied to the audit it was designed for.

The manifest file itself is not compared byte for byte: it contains the Phase 01 run time, which changes on every run. Its checksum is recorded in this phase's run manifest for lineage.

All CSVs are read as text with automatic missing-value detection switched off, so identifiers and tokens such as `NA` keep their exact form.

In [2]:
here = Path.cwd().resolve()
project_root = next((p for p in (here, *here.parents)
                     if (p / "README.md").is_file() and (p / "environment.yml").is_file() and (p / "notebooks").is_dir()), None)
if project_root is None:
    raise FileNotFoundError("Project folder not found. Open this notebook from inside the hydrogel-bioink-data-curation repository.")
audit_dir, raw_dir, output_dir = project_root / AUDIT_REL, project_root / RAW_REL, project_root / OUT_REL
zip_path = raw_dir / "ALG-Ph_HA-Ph_rheology_data.zip"
image_path = raw_dir / "image_index.csv"
manifest_path = audit_dir / "audit_run_manifest.json"
record("P02-01", "Project folder located", True, f"repository root found ('{project_root.name}')")

missing = [p.relative_to(project_root).as_posix() for p in [zip_path, image_path, manifest_path] if not p.is_file()]
record("P02-02", "Source files and Phase 01 manifest present", not missing,
       "present" if not missing else "missing: " + ", ".join(missing) + " — run scripts/download_data.py and Phase 01 first")

phase1_manifest = json.loads(manifest_path.read_text(encoding="utf-8-sig"))
verified_paths = {manifest_path: sha256(manifest_path)}
verification_rows, lineage_problems = [], []
for section, folder in [("input_sha256", raw_dir), ("export_sha256", audit_dir)]:
    for name, expected in phase1_manifest[section].items():
        path = folder / name
        actual = sha256(path) if path.is_file() else None
        if actual != expected:
            lineage_problems.append(name if actual else name + " (missing)")
            continue
        verified_paths[path] = actual
        verification_rows.append(dict(relative_path=path.relative_to(project_root).as_posix(), sha256=actual, status="matched"))
record("P02-03", "Phase 01 files match its run manifest", not lineage_problems,
       f"{len(verification_rows)} files matched" if not lineage_problems else "changed or missing since Phase 01 ran: " + ", ".join(lineage_problems))
input_verification = pd.DataFrame(verification_rows)

differs = [n for n, h in PHASE01_REFERENCE_SHA256.items() if phase1_manifest["export_sha256"].get(n) != h]
record("P02-04", "Phase 01 tables identical to the reference audit", not differs and len(phase1_manifest["export_sha256"]) == 11,
       "11 of 11 tables identical" if not differs else "differ: " + ", ".join(differs) + " — review the changed audit before using this schema")

audit_tables = {name: pd.read_csv(audit_dir / name, dtype=str, keep_default_na=False, encoding="utf-8-sig")
                for name in phase1_manifest["export_sha256"]}
workbook_inventory = audit_tables["workbook_structure_inventory.csv"]
column_profile = audit_tables["measurement_column_profile.csv"]
axis_profile = audit_tables["measurement_axis_profile.csv"]
phase1_issues = audit_tables["audit_issue_log.csv"]

[PASS] P02-01: repository root found ('hydrogel-github-main')
[PASS] P02-02: present
[PASS] P02-03: 13 files matched
[PASS] P02-04: 11 of 11 tables identical


## 3. Literature evidence and its scope

Statements from the publication ([Zhang, Elvitigala and Sakai, 2026](https://doi.org/10.1080/17452759.2026.2671497)), its supplementary material and the project documents are recorded with their scope and the schema decision each one supports. Study-level methods are context; they are not copied into individual rows as if measured there.

*Scientific note:* the paper reports amplitude sweeps at 1 Hz and a shear-rate range of 0.01–1000 s⁻¹. Both are compared with the measured axes in section 10 rather than assumed. A rheology record of an ink is not automatically a crosslinked gel: the paper separates ink characterisation from printing and light-driven crosslinking.

In [3]:
evidence_register = pd.DataFrame([
    ["main_publication", "Abstract; Section 4", "The study concerns acellular hydrogel printing.", "study", "Do not claim measured cell viability or tissue function."],
    ["main_publication", "Section 5.1", "ALG-Ph and HA-Ph inks are described in PBS at pH 7.4; study concentration ranges are given in % (w/v).", "study", "Do not populate per-file concentrations or batch chemistry from an unverified filename convention."],
    ["main_publication", "Section 5.1", "Apparent viscosity is measured with a HAAKE MARS III rheometer and 40 mm parallel plates; the reported shear-rate range is 0.01–1000 s^-1 over 360 s.", "study method", "Compare with observed axes; do not discard lower-rate workbook points to enforce the prose range."],
    ["main_publication", "Section 5.1", "G-prime and G-double-prime are reported from oscillation-amplitude sweeps at 1 Hz, 0.5 mm gap and 22.5 degrees Celsius; the text refers to the LVR.", "study method", "Use as protocol context; retain observed strain and temperature. Do not certify every point as linear-viscoelastic."],
    ["main_publication", "Section 5.2", "The authors report 45 points for each fitted rheological parameter used in their SVR dataset.", "authors' modelling dataset", "Do not equate that modelling population with 46 source files per measurement family or force them to match."],
    ["main_publication", "Section 5.3", "Printing uses Ru/SPS-containing inks under 450 nm illumination; light settings span 0–100%, with a reported irradiance range of 0.001–13.571 W/m^2.", "study printing method", "Supports a percentage-setting interpretation, not a verified CSV field definition or a linear calibration."],
    ["supplementary_material", "Figure S3 caption", "Spectral irradiance is presented for the 450 nm light source at settings from 0 to 100%.", "figure / apparatus", "No digitisation or CSV-to-irradiance conversion is performed."],
    ["supplementary_material", "Figures S1 and S5 captions", "Blueprint designs and model-evaluation printability metrics are described.", "figure / authors' analysis", "Do not assign these metrics to image-index rows; the seven-column CSV does not contain those scores."],
    ["README_Phase1", "Phase 2; Current checkpoint", "Define entities, field mappings, units, missingness and evidence requirements from the completed audit.", "project scope", "Preserve unresolved links and distinguish this prototype from later standardisation and modelling."],
    ["working_prompt", "Source-first; experimental hierarchy; rerun requirements", "Preserve raw data and source lineage; explain science; use fresh-session execution and explicit uncertainty.", "project method", "Technical keys are source locators, not invented sample identifiers."]
], columns=["document", "source_location", "source_statement", "scope", "schema_decision"])
print(len(evidence_register), 'evidence statements recorded')

10 evidence statements recorded


## 4. Entities, row granularity and keys

Three linked tables describe the rheology data at increasing detail: a **worksheet**, a **measurement point** (one source row) and a **source cell**. Each key is built from source coordinates: `member_path + sheet`, then `+ source_excel_row`, then `+ source_column_position`. Image records are keyed by their CSV record number. Four physical entities are documented but not created, because the source does not identify them.

*Scientific note:* the 50 points on one sweep are repeated observations of one acquisition, not 50 materials. Two files with the same name are not necessarily the same specimen.

In [4]:
entity_schema = pd.DataFrame([
    ["worksheet_register", "one source worksheet", "member_path + sheet", "verified archive snapshot", "Digital measurement record; physical experimental identity unresolved"],
    ["measurement_points", "one nonblank source data row", "member_path + sheet + source_excel_row", "worksheet_register", "Repeated acquisition point; not an independent sample"],
    ["measurement_values", "one cell at a measurement point", "member_path + sheet + source_excel_row + source_column_position", "measurement_points", "Preserves both acquisition context and response values"],
    ["image_records", "one image-index CSV data record", "source_csv_record", "verified image-index snapshot", "Image metadata; no verified image, construct or printing-run identity"],
], columns=["table", "row_granularity", "primary_key", "parent_scope", "scientific_boundary"])
deferred_entities = pd.DataFrame([
    ["formulation", "Source labels retained; validated composition mapping is not yet established."],
    ["physical_sample", "No verified sample identifier or sample-to-measurement relationship."],
    ["printing_event", "No verified run identifier linking rheology and image records."],
    ["printed_construct", "Image IDs do not establish construct identity or repeat imaging."],
], columns=["entity", "reason_not_instantiated"])
display(entity_schema)
display(deferred_entities)

,table,row_granularity,primary_key,parent_scope,scientific_boundary
0,worksheet_register,one source worksheet,member_path + sheet,verified archive snapshot,Digital measurement record; physical experimental identity unresolved
1,measurement_points,one nonblank source data row,member_path + sheet + source_excel_row,worksheet_register,Repeated acquisition point; not an independent sample
2,measurement_values,one cell at a measurement point,member_path + sheet + source_excel_row + source_column_position,measurement_points,Preserves both acquisition context and response values
3,image_records,one image-index CSV data record,source_csv_record,verified image-index snapshot,"Image metadata; no verified image, construct or printing-run identity"


,entity,reason_not_instantiated
0,formulation,Source labels retained; validated composition mapping is not yet established.
1,physical_sample,No verified sample identifier or sample-to-measurement relationship.
2,printing_event,No verified run identifier linking rheology and image records.
3,printed_construct,Image IDs do not establish construct identity or repeat imaging.


## 5. Source-to-schema mapping and unit policy

Each of the 13 source headers is mapped to a defined field, type, unit and conversion policy; the code stops if any observed header is unmapped. The 7 image-index columns are mapped the same way.

| Policy | Applies to |
|---|---|
| **identity** — value kept as reported | time, stress, G′, G″, frequency, temperature, shear rate, fractional strain |
| **divide by 100** — previewed in section 9, applied in Phase 03 | `ɣ in %` (percentage strain) |
| **deferred** — unit token `mPas` kept; no conversion | η and \|η\*\| |
| **retain** — text label | `SegIndex` |

*Scientific note:* shear stress and shear rate are not viscosity, and complex viscosity (oscillatory) is not steady-shear viscosity even though both use the same unit. If `mPas` is confirmed as mPa·s, the factor to Pa·s would be 0.001; that conversion is not applied here.

In [5]:
field_mapping = pd.DataFrame([
    ["SegIndex", "segment_index_source", "string", "not applicable", "retain", "source acquisition label; detailed segmentation semantics unresolved"],
    ["t in s", "time_s", "number", "s", "identity", "source time coordinate"],
    ["t_seg in s", "segment_time_s", "number", "s", "identity", "source segment-time coordinate; zero-time convention unresolved"],
    ["τ in Pa", "shear_stress_pa", "number", "Pa", "identity", "stress under the recorded test mode"],
    ["G' in Pa", "storage_modulus_pa", "number", "Pa", "identity", "elastic storage contribution under recorded oscillation conditions"],
    ['G" in Pa', "loss_modulus_pa", "number", "Pa", "identity", "viscous loss contribution under recorded oscillation conditions"],
    ["|η*| in mPas", "complex_viscosity_source", "number", "mPas", "deferred", "oscillatory complex viscosity; source unit token retained"],
    ["ɣ in -", "strain_fraction", "number", "-", "identity", "dimensionless strain fraction as labelled"],
    ["ɣ in %", "strain_fraction", "number", "%", "divide by 100", "percent strain converted only in the explicit preview"],
    ["f in Hz", "frequency_hz", "number", "Hz", "identity", "oscillation frequency, not angular frequency"],
    ["T in °C", "temperature_c", "number", "°C", "identity", "recorded temperature; not replaced by a nominal set point"],
    ["ɣ̇ in 1/s", "shear_rate_per_s", "number", "1/s", "identity", "steady-shear rate"],
    ["η in mPas", "apparent_viscosity_source", "number", "mPas", "deferred", "steady-shear apparent viscosity; source unit token retained"],
], columns=["source_header", "canonical_field", "logical_type", "source_unit", "conversion_policy", "definition"])
observed_headers = set(audit_tables["source_header_inventory.csv"]["source_header"])
require(set(field_mapping.source_header) == observed_headers, "Field map does not cover exactly the audited source headers.")
require(field_mapping.source_header.is_unique, "Source headers must have a unique mapping.")
image_mapping = pd.DataFrame([
    ["image_id", "string", "Source image label; preserve as text, including any leading zeros.", "none"],
    ["date", "string", "Source date text; validate calendar format without asserting the event it dates.", "none"],
    ["ink_type_ph", "string", "Source ink label; no concentration extraction or formulation foreign key yet.", "unresolved composition mapping"],
    ["shape", "string", "Source shape category; not a fidelity or performance score.", "none"],
    ["intensity", "string", "Source intensity token; literature supports a percentage-setting interpretation only provisionally.", "unresolved CSV unit definition"],
    ["completeness_sam", "string", "Source annotation; retain the literal token and unknown blanks.", "annotation semantics unresolved"],
    ["contain_sps", "string", "Source reagent annotation; blank does not establish absence or presence.", "annotation semantics unresolved"],
], columns=["source_column", "logical_type", "definition", "semantic_limit"])
field_mapping

,source_header,canonical_field,logical_type,source_unit,conversion_policy,definition
0,SegIndex,segment_index_source,string,not applicable,retain,source acquisition label; detailed segmentation semantics unresolved
1,t in s,time_s,number,s,identity,source time coordinate
2,t_seg in s,segment_time_s,number,s,identity,source segment-time coordinate; zero-time convention unresolved
3,τ in Pa,shear_stress_pa,number,Pa,identity,stress under the recorded test mode
4,G' in Pa,storage_modulus_pa,number,Pa,identity,elastic storage contribution under recorded oscillation conditions
5,"G"" in Pa",loss_modulus_pa,number,Pa,identity,viscous loss contribution under recorded oscillation conditions
6,|η*| in mPas,complex_viscosity_source,number,mPas,deferred,oscillatory complex viscosity; source unit token retained
7,ɣ in -,strain_fraction,number,-,identity,dimensionless strain fraction as labelled
8,ɣ in %,strain_fraction,number,%,divide by 100,percent strain converted only in the explicit preview
9,f in Hz,frequency_hz,number,Hz,identity,"oscillation frequency, not angular frequency"


## 6. Represent every source cell with its coordinates

Each workbook is read from the archive in memory and checked against Phase 01: member checksum, sheet names, headers, row counts and blank rows must all agree, and formula or Excel-error cells stop the notebook. Every non-blank row becomes a measurement point and every cell becomes a source-cell record holding the original value (as JSON, so numbers, text and empty cells stay distinct), its unit and a parse status. The parsed number is an extra field; it never replaces the source value.

The measurement family (`oscillatory_moduli` or `steady_shear_viscosity`) is assigned from the headers present, not from the folder name.

In [6]:
map_by_header = field_mapping.set_index("source_header").to_dict("index")
archive_inventory = audit_tables["archive_member_inventory.csv"]
member_hashes = dict(zip(archive_inventory.member_path, archive_inventory.sha256))
worksheet_rows, point_rows, value_rows = [], [], []
with zipfile.ZipFile(zip_path) as archive:
    require(len(archive.namelist()) == len(set(archive.namelist())), "Duplicate archive member names.")
    expected_members = set(workbook_inventory.member_path)
    actual_members = {n for n in archive.namelist() if n.lower().endswith(".xlsx")}
    require(actual_members == expected_members, "Workbook coverage differs from Phase 1.")
    for member, group in workbook_inventory.groupby("member_path", sort=True):
        payload = archive.read(member)
        require(hashlib.sha256(payload).hexdigest() == member_hashes[member], f"Member checksum mismatch: {member}")
        wb = openpyxl.load_workbook(io.BytesIO(payload), read_only=True, data_only=False)
        try:
            require(set(wb.sheetnames) == set(group.sheet), f"Sheet coverage changed: {member}")
            for expected in group.itertuples(index=False):
                ws = wb[expected.sheet]
                rows = list(ws.iter_rows())
                header_row = int(expected.candidate_header_row)
                require(header_row == 1, "This checkpoint expects headers in Excel row 1.")
                headers = ["" if c.value is None else str(c.value) for c in rows[0]]
                require(headers == json.loads(expected.headers_json), f"Header mismatch: {member} / {ws.title}")
                require(len(headers) == len(set(headers)) and all(headers), "Blank or duplicate headers.")
                require(all(c.data_type not in {"f", "e"} for row in rows for c in row), "Formula or Excel-error cell requires review.")
                active = [(i, row) for i, row in enumerate(rows[1:], 2) if not all(is_blank(c.value) for c in row)]
                require(len(active) == int(expected.data_rows), f"Point-count mismatch: {member}")
                require(len(rows)-1-len(active) == int(expected.blank_data_rows), "Blank-row count changed.")
                family = ("oscillatory_moduli" if {"G' in Pa", 'G" in Pa', "f in Hz"}.issubset(headers)
                          else "steady_shear_viscosity" if {"η in mPas", "ɣ̇ in 1/s"}.issubset(headers) else "unresolved")
                worksheet_rows.append(dict(member_path=member, sheet=ws.title,
                    source_stem=PurePosixPath(member).stem, member_sha256=member_hashes[member],
                    measurement_family=family, header_row=header_row, point_count=len(active),
                    physical_sample_link_status="unresolved"))
                for excel_row, row in active:
                    point_rows.append(dict(member_path=member, sheet=ws.title, source_excel_row=excel_row))
                    for position, (header, cell) in enumerate(zip(headers, row), 1):
                        value = cell.value
                        source_json = json.dumps(value, ensure_ascii=False, allow_nan=False)
                        if is_blank(value):
                            numeric, status = np.nan, "blank_unknown"
                        elif map_by_header[header]["logical_type"] == "string":
                            numeric, status = np.nan, "text_preserved"
                        else:
                            try:
                                numeric = float(value)
                                status = "numeric" if np.isfinite(numeric) else "nonfinite"
                                if isinstance(value, bool):
                                    status = "invalid_boolean"
                            except (ValueError, TypeError):
                                numeric, status = np.nan, "nonnumeric"
                        value_rows.append(dict(member_path=member, sheet=ws.title, source_excel_row=excel_row,
                            source_column_position=position, source_header=header, source_value_json=source_json,
                            source_unit=map_by_header[header]["source_unit"], numeric_value=numeric,
                            parse_status=status))
        finally:
            wb.close()
worksheet_register = pd.DataFrame(worksheet_rows)
measurement_points = pd.DataFrame(point_rows)
measurement_values = pd.DataFrame(value_rows)
record("P02-05", "Every worksheet, point and cell represented",
       len(worksheet_register) == EXPECTED["worksheets"] and len(measurement_points) == EXPECTED["points"] and len(measurement_values) == EXPECTED["cells"],
       f"{len(worksheet_register)} worksheets, {len(measurement_points):,} points, {len(measurement_values):,} cells")
worksheet_register.groupby("measurement_family").agg(worksheets=("sheet", "size"), points=("point_count", "sum")).reset_index()

[PASS] P02-05: 92 worksheets, 4,600 points, 39,100 cells


,measurement_family,worksheets,points
0,oscillatory_moduli,46,2300
1,steady_shear_viscosity,46,2300


## 7. Image metadata and missing values

Each image-index record keeps its source values as text, gains a `_status` column per field (`reported` or `unknown_blank`) and is joined one-to-one to its Phase 01 expected file name. The missing-value rules state how each kind of absence is treated: a blank is unknown, never "no" or zero.

In [7]:
with image_path.open(encoding="utf-8-sig", newline="") as handle:
    image_csv = list(csv.reader(handle))
image_headers = image_csv[0]
require(image_headers == image_mapping.source_column.tolist(), "Image-index headers differ from the schema.")
require(all(len(r) == len(image_headers) for r in image_csv[1:]), "Malformed CSV record width.")
image_records = pd.DataFrame(image_csv[1:], columns=image_headers)
image_records.insert(0, "source_csv_record", range(2, len(image_records)+2))
for column in image_headers:
    image_records[column + "_status"] = np.where(image_records[column].str.strip().eq(""), "unknown_blank", "reported")
expectations = audit_tables["image_filename_expectations.csv"].copy()
expectations["source_csv_record"] = expectations.source_csv_record.astype(int)
image_records = image_records.merge(expectations[["source_csv_record", "image_id", "expected_filename", "existence_status"]],
                                    on=["source_csv_record", "image_id"], how="left", validate="one_to_one")
require(image_records.expected_filename.notna().all(), "An image-index record lacks its Phase 1 filename expectation.")
image_missingness = pd.DataFrame([dict(source_column=c, rows=len(image_records),
    unknown_blank=int(image_records[c + "_status"].eq("unknown_blank").sum())) for c in image_headers])
observed_categories = audit_tables["image_index_value_counts.csv"].copy()
missing_value_rules = pd.DataFrame([
    ["blank source cell / CSV token", "unknown", "Preserve the original scalar/token and mark observed blankness."],
    ["literal no", "reported source annotation", "Retain no; do not infer the annotation's full scope or calibration."],
    ["nonnumeric measurement", "unparseable", "Retain the source scalar and flag; do not impute."],
    ["quantity absent from a worksheet layout", "structurally absent", "No source-cell row is invented for an absent header."],
    ["unresolved unit or relationship", "unresolved semantics", "Keep the value or label; do not create a converted value or scientific foreign key."],
], columns=["source_condition", "interpretation", "action"])
record("P02-06", "Every image record represented with its expected file name",
       len(image_records) == EXPECTED["image_records"] and image_records.expected_filename.notna().all(),
       f"{len(image_records):,} records; " + "; ".join(f"{r.source_column} {r.unknown_blank} blank" for r in image_missingness.itertuples() if r.unknown_blank))
missing_value_rules

[PASS] P02-06: 1,031 records; completeness_sam 943 blank; contain_sps 922 blank


,source_condition,interpretation,action
0,blank source cell / CSV token,unknown,Preserve the original scalar/token and mark observed blankness.
1,literal no,reported source annotation,Retain no; do not infer the annotation's full scope or calibration.
2,nonnumeric measurement,unparseable,Retain the source scalar and flag; do not impute.
3,quantity absent from a worksheet layout,structurally absent,No source-cell row is invented for an absent header.
4,unresolved unit or relationship,unresolved semantics,Keep the value or label; do not create a converted value or scientific foreign key.


## 8. Data dictionary

Every field in the four core tables is documented with its type, definition and provenance. The code stops if any field lacks a definition, so the dictionary cannot fall out of step with the tables.

In [8]:
technical_definitions = {
 "member_path": ("string", "Exact workbook path inside the verified ZIP; worksheet-key component."),
 "sheet": ("string", "Exact source worksheet name; worksheet-key component."),
 "source_stem": ("string", "Unparsed workbook basename without extension; not a sample ID."),
 "member_sha256": ("string", "SHA-256 of workbook bytes from the verified archive."),
 "measurement_family": ("string", "Header-supported measurement family; not a complete protocol or specimen state."),
 "header_row": ("integer", "One-based source Excel header row."),
 "point_count": ("integer", "Count of nonblank source data rows; not independent sample size."),
 "physical_sample_link_status": ("string", "Unresolved physical identity; no physical-sample foreign key exists."),
 "source_excel_row": ("integer", "One-based original Excel row coordinate; point-key component."),
 "source_column_position": ("integer", "One-based original Excel column coordinate; cell-key component."),
 "source_header": ("string", "Exact source column header; foreign key to field_mapping.source_header."),
 "source_value_json": ("string", "JSON-encoded source scalar; retains number/text/null distinction."),
 "source_unit": ("string", "Exact header unit token, or not applicable for a label."),
 "numeric_value": ("number or null", "Parsed source numeric value with no unit conversion; consult parse_status."),
 "parse_status": ("string", "numeric, text_preserved, blank_unknown, nonnumeric, nonfinite or invalid_boolean."),
 "source_csv_record": ("integer", "One-based CSV record number, counting header as record 1; not a physical line counter."),
 "expected_filename": ("string", "Phase 1 filename expectation; file existence is not asserted."),
 "existence_status": ("string", "Phase 1 image-existence status carried forward verbatim."),
}
core_tables = {"worksheet_register": worksheet_register, "measurement_points": measurement_points,
               "measurement_values": measurement_values, "image_records": image_records}
dictionary_rows = []
for table, frame in core_tables.items():
    for field in frame.columns:
        if table == "image_records" and field in image_headers:
            description = image_mapping.set_index("source_column").loc[field, "definition"]
            dtype, origin = "string", "source-reported token"
        elif table == "image_records" and field.endswith("_status"):
            dtype, description, origin = "string", "reported or unknown_blank for " + field[:-7], "derived missingness status"
        else:
            require(field in technical_definitions, f"Undocumented field: {table}.{field}")
            dtype, description = technical_definitions[field]
            origin = "source locator, representation or explicitly derived metadata"
        dictionary_rows.append(dict(table=table, field=field, logical_type=dtype, definition=description,
            provenance=origin, unit_policy="See exact source_header mapping for measurement values; no implicit conversion.",
            missingness_policy="Source blanks retained; derived nulls require the associated parsing/status field."))
data_dictionary = pd.DataFrame(dictionary_rows)
require(not data_dictionary.duplicated(["table", "field"]).any(), "Duplicate dictionary definitions.")
record("P02-07", "Data dictionary covers every field", len(data_dictionary) == EXPECTED["dictionary_fields"],
       f"{len(data_dictionary)} fields documented across {data_dictionary.table.nunique()} tables")

[PASS] P02-07: 37 fields documented across 4 tables


## 9. Quality-control rules and structural checks

Fourteen QC rules are defined with a severity: **ERROR** rules stop the notebook; **WARNING** rules produce findings that are retained for review; **INFO** documents a tested calculation. The structural checks test unique keys, parent-child links with no orphans or duplicates, and exact agreement of every count with Phase 01. All structural and reconciliation checks are exported together as `validation_summary.csv`.

In [9]:
qc_rules = pd.DataFrame([
 ["CHECKPOINT_HASH", "ERROR", "Input and audit hashes must match the reviewed snapshot.", "Stop on mismatch."],
 ["SOURCE_COVERAGE", "ERROR", "Every audited workbook, sheet, point and cell must be represented.", "Stop on missing or additional records."],
 ["PRIMARY_KEY", "ERROR", "Natural source-coordinate keys must be unique and nonblank.", "Stop; do not deduplicate silently."],
 ["FOREIGN_KEY", "ERROR", "Every child record must have exactly one source parent.", "Stop on orphan or multiplication."],
 ["NUMERIC_PARSE", "WARNING", "Every measurement value should be finite numeric or explicitly flagged.", "Keep original scalar and parse status."],
 ["NONPOSITIVE_RESPONSE", "WARNING", "Viscosity and moduli <= 0 need review before logs or fitting.", "Retain; cause is not established."],
 ["ACQUISITION_AXIS", "WARNING", "Nonpositive frequency/shear rate or negative strain needs review.", "Retain and inspect test context."],
 ["DUPLICATE_ROW", "WARNING", "Identical full rows in a worksheet need contextual review.", "Retain; do not assume an export error."],
 ["IMAGE_ID", "WARNING", "Blank or repeated image labels need review.", "Retain source record coordinates."],
 ["IMAGE_DATE", "WARNING", "Source dates should parse as YYYY/MM/DD calendar dates.", "Retain original text."],
 ["IMAGE_BLANK", "WARNING", "Blank image metadata remains unknown.", "Do not fill with zero, yes or no."],
 ["IMAGE_VOCABULARY", "ERROR", "Categories and counts must match the Phase 1 snapshot.", "Stop on unexplained drift."],
 ["UNRESOLVED_SEMANTICS", "WARNING", "Unit, annotation and physical-link uncertainties remain explicit.", "Defer unsupported transformations and joins."],
 ["STRAIN_PREVIEW", "INFO", "Percent-to-fraction arithmetic must be reversible within floating-point tolerance.", "Preview only; keep the source scalar."],
], columns=["rule", "severity", "criterion", "action"])
checks = []
def check(name, passed, evidence):
    checks.append(dict(check=name, status="PASS" if bool(passed) else "FAIL", evidence=str(evidence)))
    require(passed, f"Structural check failed: {name}. {evidence}")
worksheet_key = ["member_path", "sheet"]
point_key = worksheet_key + ["source_excel_row"]
cell_key = point_key + ["source_column_position"]
for name, frame, key in [("worksheet", worksheet_register, worksheet_key),
                         ("point", measurement_points, point_key),
                         ("cell", measurement_values, cell_key),
                         ("image record", image_records, ["source_csv_record"])]:
    check(name + " primary key", not frame.duplicated(key).any() and not frame[key].isna().any().any()
          and frame[key].astype(str).apply(lambda s: s.str.strip().ne("")).all().all(), len(frame))
for name, child, parent, key in [("point to worksheet", measurement_points, worksheet_register, worksheet_key),
                                 ("cell to point", measurement_values, measurement_points, point_key)]:
    linked = child[key].merge(parent[key], on=key, how="left", validate="many_to_one", indicator=True)
    check(name, linked._merge.eq("both").all() and len(linked) == len(child), "No orphan or multiplied records")
check("worksheet count", len(worksheet_register) == phase1_manifest["worksheets_audited"], len(worksheet_register))
check("point count", len(measurement_points) == phase1_manifest["measurement_rows"], len(measurement_points))
check("image count", len(image_records) == phase1_manifest["image_index_rows"], len(image_records))
check("cell count", len(measurement_values) == column_profile.observed_rows.astype(int).sum(), len(measurement_values))
cell_counts = measurement_values.groupby(worksheet_key + ["source_column_position"]).size().rename("actual").reset_index()
expected_counts = column_profile[["member_path", "sheet", "column_position", "observed_rows"]].copy()
expected_counts["source_column_position"] = expected_counts.pop("column_position").astype(int)
count_comparison = cell_counts.merge(expected_counts, on=worksheet_key+["source_column_position"], how="outer", validate="one_to_one")
check("per-column coverage", count_comparison.actual.eq(count_comparison.observed_rows.astype(int)).all(), len(count_comparison))
check("dictionary coverage", len(data_dictionary) == sum(len(t.columns) for t in core_tables.values()), len(data_dictionary))
check("source-header mapping", measurement_values.source_header.isin(field_mapping.source_header).all(), "All source headers mapped")
qc_rules

,rule,severity,criterion,action
0,CHECKPOINT_HASH,ERROR,Input and audit hashes must match the reviewed snapshot.,Stop on mismatch.
1,SOURCE_COVERAGE,ERROR,"Every audited workbook, sheet, point and cell must be represented.",Stop on missing or additional records.
2,PRIMARY_KEY,ERROR,Natural source-coordinate keys must be unique and nonblank.,Stop; do not deduplicate silently.
3,FOREIGN_KEY,ERROR,Every child record must have exactly one source parent.,Stop on orphan or multiplication.
4,NUMERIC_PARSE,WARNING,Every measurement value should be finite numeric or explicitly flagged.,Keep original scalar and parse status.
5,NONPOSITIVE_RESPONSE,WARNING,Viscosity and moduli <= 0 need review before logs or fitting.,Retain; cause is not established.
6,ACQUISITION_AXIS,WARNING,Nonpositive frequency/shear rate or negative strain needs review.,Retain and inspect test context.
7,DUPLICATE_ROW,WARNING,Identical full rows in a worksheet need contextual review.,Retain; do not assume an export error.
8,IMAGE_ID,WARNING,Blank or repeated image labels need review.,Retain source record coordinates.
9,IMAGE_DATE,WARNING,Source dates should parse as YYYY/MM/DD calendar dates.,Retain original text.


## 10. Value-level QC without silent cleaning

Each source cell is checked for parse problems, non-positive responses (G′, G″, η, |η\*|) and non-positive or negative acquisition axes. Repeated rows, blank image fields, image IDs and dates are checked too. Image categories and blank counts must match Phase 01 exactly, and the non-positive response cells must be the same cells Phase 01 flagged. Every finding is recorded with its source location; no value is changed or removed.

*Scientific note:* non-positive moduli or viscosities are physically unusual and may reflect the instrument's measurement limit at low strain or shear rate. The cause is not established here, so the values are kept and flagged.

In [10]:
findings = []
def finding(rule, source, location, evidence, action, severity="WARNING"):
    findings.append(dict(rule=rule, severity=severity, source=source, location=location,
                         evidence=str(evidence), action=action))
responses = {"G' in Pa", 'G" in Pa', "η in mPas", "|η*| in mPas"}
for row in measurement_values.itertuples(index=False):
    loc = f"{row.sheet}; row {row.source_excel_row}; column {row.source_column_position}; {row.source_header}"
    if row.parse_status not in {"numeric", "text_preserved"}:
        finding("NUMERIC_PARSE", row.member_path, loc, row.source_value_json, "Retain source scalar; review type or missingness.")
    if row.parse_status == "numeric":
        if row.source_header in responses and row.numeric_value <= 0:
            finding("NONPOSITIVE_RESPONSE", row.member_path, loc, row.numeric_value, "Retain; assess measurement limits before modelling.")
        if ((row.source_header in {"ɣ̇ in 1/s", "f in Hz"} and row.numeric_value <= 0) or
            (row.source_header in {"ɣ in -", "ɣ in %"} and row.numeric_value < 0)):
            finding("ACQUISITION_AXIS", row.member_path, loc, row.numeric_value, "Retain and review acquisition context.")
wide_raw = measurement_values.pivot(index=point_key, columns="source_column_position", values="source_value_json")
for key, frame in wide_raw.groupby(level=[0, 1], sort=True):
    for duplicate in frame[frame.duplicated(keep=False)].index:
        finding("DUPLICATE_ROW", key[0], f"{key[1]}; row {duplicate[2]}", "Identical full source row", "Retain pending contextual review.")
for row in image_missingness.itertuples(index=False):
    if row.unknown_blank:
        finding("IMAGE_BLANK", "image_index.csv", row.source_column, f"{row.unknown_blank}/{row.rows} unknown blanks", "Keep unknown; do not infer annotation outcomes.")
bad_ids = image_records.image_id.str.strip().eq("") | image_records.image_id.duplicated(keep=False)
for row in image_records.loc[bad_ids].itertuples(index=False):
    finding("IMAGE_ID", "image_index.csv", f"record {row.source_csv_record}", row.image_id, "Retain; investigate source image identity.")
valid_date_shape = image_records.date.str.fullmatch(r"\d{4}/\d{2}/\d{2}")
parsed_dates = pd.to_datetime(image_records.date, format="%Y/%m/%d", errors="coerce")
for row in image_records.loc[~valid_date_shape | parsed_dates.isna()].itertuples(index=False):
    finding("IMAGE_DATE", "image_index.csv", f"record {row.source_csv_record}", row.date, "Retain original date; review format.")
for column in observed_categories.column.unique():
    expected = observed_categories.loc[observed_categories.column.eq(column)]
    expected_counts = {r.source_value: int(r.count) for r in expected.itertuples(index=False)}
    actual_counts = image_records[column].value_counts(dropna=False).to_dict()
    check("category counts: " + column, actual_counts == expected_counts, sum(actual_counts.values()))
for row in audit_tables["image_index_column_profile.csv"].itertuples(index=False):
    actual_blank = int(image_records[row.column + "_status"].eq("unknown_blank").sum())
    check("image missingness: " + row.column, actual_blank == int(row.blank_count), actual_blank)
phase1_nonpositive = set()
for row in phase1_issues.loc[phase1_issues.issue_code.eq("NONPOSITIVE_RHEOLOGY_VALUE")].itertuples(index=False):
    for excel_row in json.loads(row.evidence.split(":", 1)[1].strip()):
        phase1_nonpositive.add((row.source, row.sheet, excel_row, row.column))
nonpositive_cells = measurement_values.loc[measurement_values.source_header.isin(responses)
    & measurement_values.parse_status.eq("numeric") & measurement_values.numeric_value.le(0)].copy()
actual_nonpositive = set(nonpositive_cells[["member_path", "sheet", "source_excel_row", "source_header"]].itertuples(index=False, name=None))
check("nonpositive response locations agree with Phase 1", actual_nonpositive == phase1_nonpositive, len(actual_nonpositive))
value_qc_findings = pd.DataFrame(findings, columns=["rule", "severity", "source", "location", "evidence", "action"])
n_nonpositive_ws = len(nonpositive_cells[worksheet_key].drop_duplicates())
record("P02-08", "Non-positive responses match Phase 01",
       len(nonpositive_cells) == EXPECTED["nonpositive_cells"] and n_nonpositive_ws == EXPECTED["nonpositive_worksheets"],
       f"{len(nonpositive_cells)} cells in {n_nonpositive_ws} worksheets, same locations as Phase 01")
record("P02-09", "Value-QC findings match the reference run", len(value_qc_findings) == EXPECTED["value_findings"],
       f"{len(value_qc_findings)} findings recorded")
value_qc_findings.groupby(["rule", "severity"]).size().rename("finding_count").reset_index()

[PASS] P02-08: 20 cells in 8 worksheets, same locations as Phase 01
[PASS] P02-09: 24 findings recorded


,rule,severity,finding_count
0,ACQUISITION_AXIS,WARNING,2
1,IMAGE_BLANK,WARNING,2
2,NONPOSITIVE_RESPONSE,WARNING,20


## 11. Strain conversion preview and measurement context

The percent-to-fraction rule (`value / 100`) is applied in a separate preview table and checked by reversing it. The source values are not changed; Phase 03 applies the rule. The context summary compares the measured axes with the published methods.

*Scientific note:* a fraction of 0.01 equals 1 % strain. Converting units does not identify the linear viscoelastic region.

In [11]:
strain_preview = measurement_values.loc[measurement_values.source_header.isin(["ɣ in -", "ɣ in %"])].copy()
require(strain_preview.parse_status.eq("numeric").all(), "Review strain parsing before generating a conversion preview.")
strain_preview["conversion_factor"] = np.where(strain_preview.source_unit.eq("%"), 0.01, 1.0)
strain_preview["standardised_value"] = strain_preview.numeric_value * strain_preview.conversion_factor
strain_preview["standardised_unit"] = "1 (strain fraction)"
strain_preview["conversion_rule"] = np.where(strain_preview.source_unit.eq("%"), "source value / 100", "identity: source fraction")
strain_preview["transformation_status"] = "schema preview; source unchanged"
check("strain conversion round trip", np.allclose(strain_preview.standardised_value / strain_preview.conversion_factor,
      strain_preview.numeric_value, rtol=1e-12, atol=0), len(strain_preview))
strain_summary = strain_preview.groupby("source_unit").agg(cells=("source_unit", "size"),
    source_min=("numeric_value", "min"), source_max=("numeric_value", "max"),
    fraction_min=("standardised_value", "min"), fraction_max=("standardised_value", "max")).reset_index()
strain_summary

,source_unit,cells,source_min,source_max,fraction_min,fraction_max
0,%,100,0.005846,31.597120,0.000058,0.315971
1,-,2200,0.000071,335.218689,0.000071,335.218689


In [12]:
axis_context = axis_profile.copy()
for c in ["numeric_min", "numeric_max", "distinct_numeric_values"]:
    axis_context[c] = pd.to_numeric(axis_context[c], errors="raise")
axis_context = axis_context.merge(worksheet_register[worksheet_key+["measurement_family"]],
                                  on=worksheet_key, how="left", validate="many_to_one")
frequency_context = axis_context.loc[axis_context.source_header.eq("f in Hz")]
shear_context = axis_context.loc[axis_context.source_header.eq("ɣ̇ in 1/s")]
context_summary = axis_context.groupby(["measurement_family", "source_header"]).agg(
    worksheets=("sheet", "size"), minimum=("numeric_min", "min"), maximum=("numeric_max", "max"),
    fewest_distinct_values=("distinct_numeric_values", "min"), most_distinct_values=("distinct_numeric_values", "max")).reset_index()
display(context_summary)
context_review = pd.DataFrame([
    {"question": "Oscillation frequency", "observed": f"{len(frequency_context)} sheets; {int((frequency_context.numeric_min.eq(1) & frequency_context.numeric_max.eq(1)).sum())} fixed at 1 Hz",
     "interpretation": "Consistent with the reported amplitude-sweep method; LVR still unassigned."},
    {"question": "Shear-rate lower bound", "observed": f"{int(shear_context.numeric_min.lt(0.01).sum())} worksheets include values below 0.01 1/s",
     "interpretation": "Retain those points and review the difference from the nominal range in Section 5.1."},
    {"question": "Independent sample count", "observed": "Not established by these source-coordinate keys",
     "interpretation": "Do not use worksheet or point counts as biological or physical replication."},
])
display(context_review)

,measurement_family,source_header,worksheets,minimum,maximum,fewest_distinct_values,most_distinct_values
0,oscillatory_moduli,T in °C,46,22.497675,22.503015,26,39
1,oscillatory_moduli,f in Hz,46,1.000000,1.000000,1,1
2,oscillatory_moduli,ɣ in %,2,0.005846,31.597120,50,50
3,oscillatory_moduli,ɣ in -,44,0.000071,335.218689,50,50
4,steady_shear_viscosity,T in °C,46,22.498224,22.502344,27,37
5,steady_shear_viscosity,ɣ̇ in 1/s,46,-0.000967,1000.525635,50,50


,question,observed,interpretation
0,Oscillation frequency,46 sheets; 46 fixed at 1 Hz,Consistent with the reported amplitude-sweep method; LVR still unassigned.
1,Shear-rate lower bound,46 worksheets include values below 0.01 1/s,Retain those points and review the difference from the nominal range in Section 5.1.
2,Independent sample count,Not established by these source-coordinate keys,Do not use worksheet or point counts as biological or physical replication.


## 12. Carry forward Phase 01 issues and save the outputs

Every Phase 01 finding is carried forward unchanged, with a Phase 02 decision for each issue type; the code stops if a type has no decision. Additional review topics found while building the schema are listed separately. Before writing, the notebook confirms that no earlier check failed and that every input is unchanged. The 26 tables and the run manifest replace any earlier run in `data/metadata/schema_and_quality_control/`.

In [13]:
phase1_issue_carry_forward = phase1_issues.copy(deep=True)
decision_by_code = {
 "DATASET_LICENCE_UNRESOLVED": ("open", "Dataset-specific reuse terms are not established by the article licence. No live licence reassessment in this notebook."),
 "IMAGE_FILES_NOT_AUDITED": ("open", "Expected filenames retained; no physical image files inspected."),
 "IMAGE_METADATA_BLANK": ("open; represented", "Explicit unknown_blank status introduced; no imputation or yes/no inference."),
 "INTENSITY_UNIT_UNRESOLVED": ("partly clarified; open", "Section 5.3 and Figure S3 support a percentage-setting interpretation; exact CSV semantics and calibration remain unverified."),
 "MIXED_STRAIN_UNITS": ("rule defined; preview tested", "Both source representations retained; fraction/percentage arithmetic is traceable. Full standardisation remains later work."),
 "NONPOSITIVE_RHEOLOGY_VALUE": ("open; locations reconciled", "Affected cells matched to Phase 1 and retained unchanged; no causal diagnosis or exclusion."),
 "PHYSICAL_SAMPLE_LINK_UNRESOLVED": ("open", "Source-coordinate keys identify records only; no sample or run identifiers created."),
}
require(set(phase1_issues.issue_code).issubset(decision_by_code), "A Phase 1 issue category has no Phase 2 decision.")
issue_decisions = pd.DataFrame([dict(phase1_issue_id=r.issue_id, issue_code=r.issue_code,
    phase2_status=decision_by_code[r.issue_code][0], decision=decision_by_code[r.issue_code][1])
    for r in phase1_issues.itertuples(index=False)])
check("issue carry-forward", phase1_issue_carry_forward.equals(phase1_issues), len(phase1_issue_carry_forward))
filename_candidates = audit_tables["candidate_filename_correspondences.csv"].copy()
linkage_summary = pd.DataFrame([
    {"quantity": "Source stems in candidate table", "count": len(filename_candidates)},
    {"quantity": "Stems with exactly one file in each family", "count": int(filename_candidates.exact_one_each.str.lower().eq("true").sum())},
    {"quantity": "Physical-sample links confirmed by this notebook", "count": 0},
])
additional_review = pd.DataFrame([
 ["VISCOSITY_UNIT_TOKEN", "open", "mPas retained; establish instrument unit notation before production conversion to Pa·s."],
 ["LVR_SELECTION", "open", "Study method reported; per-curve linear-regime selection and representative moduli not established."],
 ["PUBLICATION_RANGE", "open", "Observed shear-rate minima extend below the nominal method range; retain and review."],
 ["MODEL_POPULATION", "open", "Authors' 45 fitted-parameter data points are not reconciled to the 46 files per family; no forced exclusion."],
], columns=["review_topic", "status", "next_action"])
linkage_summary

,quantity,count
0,Source stems in candidate table,62
1,Stems with exactly one file in each family,30
2,Physical-sample links confirmed by this notebook,0


In [14]:
require(all(c["status"] != "FAIL" for c in phase_checks), "Outputs not written: an earlier check failed.")
for path, original_hash in verified_paths.items():
    check("input preserved: " + path.name, sha256(path) == original_hash, "SHA-256 unchanged")
validation_summary = pd.DataFrame(checks)
exports = {
 "input_verification.csv": input_verification,
 "reviewed_inputs.csv": reviewed_inputs,
 "literature_evidence_register.csv": evidence_register,
 "entity_schema.csv": entity_schema,
 "deferred_entities.csv": deferred_entities,
 "source_to_schema_mapping.csv": field_mapping,
 "image_field_dictionary.csv": image_mapping,
 "data_dictionary.csv": data_dictionary,
 "missing_value_rules.csv": missing_value_rules,
 "worksheet_register.csv": worksheet_register,
 "measurement_points.csv": measurement_points,
 "measurement_values_source.csv": measurement_values,
 "image_records_source.csv": image_records,
 "image_missingness.csv": image_missingness,
 "image_observed_categories.csv": observed_categories,
 "quality_control_rules.csv": qc_rules,
 "validation_summary.csv": validation_summary,
 "value_qc_findings.csv": value_qc_findings,
 "strain_conversion_preview.csv": strain_preview,
 "measurement_context_summary.csv": context_summary,
 "context_review.csv": context_review,
 "phase1_issue_carry_forward.csv": phase1_issue_carry_forward,
 "phase2_issue_decisions.csv": issue_decisions,
 "additional_review_items.csv": additional_review,
 "filename_candidates_unconfirmed.csv": filename_candidates,
 "linkage_summary.csv": linkage_summary,
}
output_dir.mkdir(parents=True, exist_ok=True)
for name, table in exports.items():
    target = output_dir / name
    temporary = target.with_suffix(target.suffix + ".tmp")
    table.to_csv(temporary, index=False, encoding="utf-8-sig", lineterminator="\n")
    temporary.replace(target)
run_manifest = {
 "phase": "2: schema and QC prototype", "run_utc": datetime.now(timezone.utc).isoformat(),
 "source_record": phase1_manifest["source_record"], "source_version": "v0.0.2",
 "phase1_manifest_sha256": sha256(manifest_path),
 "environment": environment.to_dict("records"),
 "verified_input_sha256": {p.relative_to(project_root).as_posix(): h for p, h in verified_paths.items()},
 "reviewed_supporting_inputs": reviewed_inputs.to_dict("records"),
 "rows": {name: len(table) for name, table in exports.items()},
 "export_sha256": {name: sha256(output_dir / name) for name in exports},
 "raw_and_phase1_files_preserved": True,
 "physical_sample_linkage": "unresolved", "image_existence": "not tested",
 "viscosity_unit_conversion": "deferred", "strain_conversion": "preview only",
 "dataset_licence": "unresolved at reviewed checkpoint", "hydrogel_model_readiness": "not established",
}
manifest_target = output_dir / "schema_qc_run_manifest.json"
temporary = manifest_target.with_suffix(".json.tmp")
temporary.write_text(json.dumps(run_manifest, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
temporary.replace(manifest_target)
n_struct_pass = int(validation_summary.status.eq("PASS").sum())
record("P02-10", "Structural and reconciliation checks", n_struct_pass == len(validation_summary) == EXPECTED["structural_checks"],
       f"{n_struct_pass} of {len(validation_summary)} passed (exported as validation_summary.csv)")
record("P02-11", "Inputs unchanged and outputs written", len(exports) == EXPECTED["exports"],
       f"{len(verified_paths)} inputs unchanged; {len(exports)} tables and schema_qc_run_manifest.json written to {OUT_REL.as_posix()}/")
differing = [n for n, h in run_manifest["export_sha256"].items() if REFERENCE_EXPORT_SHA256.get(n) != h]
record("P02-12", "Output tables identical to the reference run (SHA-256)", not differing,
       f"{len(exports) - len(differing)} of {len(exports)} tables identical" + (f"; differ: {', '.join(differing)}" if differing else ""), warn_only=True)

[PASS] P02-10: 43 of 43 passed (exported as validation_summary.csv)
[PASS] P02-11: 14 inputs unchanged; 26 tables and schema_qc_run_manifest.json written to data/metadata/schema_and_quality_control/
[PASS] P02-12: 26 of 26 tables identical


## 13. Validation summary

This phase's checks, followed by the overall result. The 43 structural checks behind P02-10 are listed individually in `validation_summary.csv`. The check table is also saved as `phase02_checks.csv`.

In [15]:
check_table = pd.DataFrame(phase_checks)
check_table.to_csv(output_dir / "phase02_checks.csv", index=False, encoding="utf-8-sig", lineterminator="\n")
counts = check_table.status.value_counts()
n_pass, n_warn, n_fail = (int(counts.get(s, 0)) for s in ("PASS", "WARN", "FAIL"))
display(check_table)
print(f"Phase 02: {n_pass} of {len(check_table)} checks passed, {n_warn} warning(s), {n_fail} failure(s).")

,check_id,description,status,detail
0,P02-01,Project folder located,PASS,repository root found ('hydrogel-github-main')
1,P02-02,Source files and Phase 01 manifest present,PASS,present
2,P02-03,Phase 01 files match its run manifest,PASS,13 files matched
3,P02-04,Phase 01 tables identical to the reference audit,PASS,11 of 11 tables identical
4,P02-05,"Every worksheet, point and cell represented",PASS,"92 worksheets, 4,600 points, 39,100 cells"
5,P02-06,Every image record represented with its expected file name,PASS,"1,031 records; completeness_sam 943 blank; contain_sps 922 blank"
6,P02-07,Data dictionary covers every field,PASS,37 fields documented across 4 tables
7,P02-08,Non-positive responses match Phase 01,PASS,"20 cells in 8 worksheets, same locations as Phase 01"
8,P02-09,Value-QC findings match the reference run,PASS,24 findings recorded
9,P02-10,Structural and reconciliation checks,PASS,43 of 43 passed (exported as validation_summary.csv)


Phase 02: 12 of 12 checks passed, 0 warning(s), 0 failure(s).


## 14. Findings

Each row summarises one result with its status label, what it means, and whether it needs attention. Details are in the exported tables named in the last column.

In [16]:
def n_rule(rule):
    return int(value_qc_findings.rule.eq(rule).sum())

fixed_1hz = int((frequency_context.numeric_min.eq(1) & frequency_context.numeric_max.eq(1)).sum())
low_shear = int(shear_context.numeric_min.lt(0.01).sum())
pct_cells = int(strain_preview.source_unit.eq("%").sum())
findings_table = pd.DataFrame([
    {"finding": f"Schema represents every source cell: {len(worksheet_register)} worksheets, {len(measurement_points):,} points, {len(measurement_values):,} cells; 13 headers and 7 image fields mapped",
     "status": "Verified", "meaning": "Each value can be traced to its archive file, sheet, Excel row and column", "attention": "No", "details": "measurement_values_source.csv"},
    {"finding": "Formulation, physical sample, printing event and printed construct are not created",
     "status": "Unresolved", "meaning": "The source has no identifiers for them; keys identify digital records only", "attention": "No action in Phase 02", "details": "deferred_entities.csv"},
    {"finding": f"Percentage strain ({pct_cells} cells) maps to strain fraction by dividing by 100; the preview reverses exactly",
     "status": "Verified", "meaning": "The conversion rule is ready; source values are unchanged until Phase 03", "attention": "No", "details": "strain_conversion_preview.csv"},
    {"finding": "Viscosity unit token 'mPas' kept as written; no conversion to Pa·s",
     "status": "Unresolved", "meaning": "Viscosity values cannot yet be compared with values reported in Pa·s", "attention": "Yes — confirm the instrument's unit notation", "details": "additional_review_items.csv"},
    {"finding": f"Non-positive G′, G″, η or |η*|: {len(nonpositive_cells)} cells in {n_nonpositive_ws} worksheets, same cells as Phase 01",
     "status": "Preliminary", "meaning": "Cause not established (possibly the measurement limit); values retained and flagged", "attention": "Yes — review before log scales or model fitting", "details": "value_qc_findings.csv"},
    {"finding": f"Non-positive shear-rate values: {n_rule('ACQUISITION_AXIS')} cells",
     "status": "Preliminary", "meaning": "Retained; a shear rate of zero or below needs acquisition context", "attention": "Yes — review with the non-positive responses", "details": "value_qc_findings.csv"},
    {"finding": f"{low_shear} viscosity worksheets include shear rates below the paper's stated 0.01 s⁻¹",
     "status": "Verified", "meaning": "Measured range differs from the published method; points retained", "attention": "No action in Phase 02", "details": "context_review.csv"},
    {"finding": f"{fixed_1hz} of {len(frequency_context)} modulus worksheets fixed at 1 Hz",
     "status": "Assumption", "meaning": "Consistent with the paper's amplitude sweeps; the linear viscoelastic region is not assigned", "attention": "No", "details": "measurement_context_summary.csv"},
    {"finding": "Image blanks recorded as unknown_blank: " + "; ".join(f"{r.source_column} {r.unknown_blank}" for r in image_missingness.itertuples() if r.unknown_blank),
     "status": "Verified", "meaning": "No blank is read as 'no', zero or a failed print", "attention": "No", "details": "image_missingness.csv"},
    {"finding": "Intensity: the paper supports a 0–100 % light-setting reading",
     "status": "Unresolved", "meaning": "Partly clarified; the CSV unit and calibration are not confirmed, so no irradiance is calculated", "attention": "Yes — confirm with the methods or authors", "details": "phase2_issue_decisions.csv"},
    {"finding": "Authors' modelling set has 45 points per parameter; the archive has 46 files per family",
     "status": "Unresolved", "meaning": "The two populations are not reconciled; no file is excluded to force a match", "attention": "No action in Phase 02", "details": "additional_review_items.csv"},
    {"finding": f"{int(linkage_summary['count'].iloc[0])} file-name stems, {int(linkage_summary['count'].iloc[1])} with one file per family; 0 physical-sample links confirmed",
     "status": "Unresolved", "meaning": "Name matches remain candidates; Phase 04 tests linkage", "attention": "No action in Phase 02", "details": "filename_candidates_unconfirmed.csv"},
])
findings_table

,finding,status,meaning,attention,details
0,"Schema represents every source cell: 92 worksheets, 4,600 points, 39,100 cells; 13 headers and 7 image fields mapped",Verified,"Each value can be traced to its archive file, sheet, Excel row and column",No,measurement_values_source.csv
1,"Formulation, physical sample, printing event and printed construct are not created",Unresolved,The source has no identifiers for them; keys identify digital records only,No action in Phase 02,deferred_entities.csv
2,Percentage strain (100 cells) maps to strain fraction by dividing by 100; the preview reverses exactly,Verified,The conversion rule is ready; source values are unchanged until Phase 03,No,strain_conversion_preview.csv
3,Viscosity unit token 'mPas' kept as written; no conversion to Pa·s,Unresolved,Viscosity values cannot yet be compared with values reported in Pa·s,Yes — confirm the instrument's unit notation,additional_review_items.csv
4,"Non-positive G′, G″, η or |η*|: 20 cells in 8 worksheets, same cells as Phase 01",Preliminary,Cause not established (possibly the measurement limit); values retained and flagged,Yes — review before log scales or model fitting,value_qc_findings.csv
5,Non-positive shear-rate values: 2 cells,Preliminary,Retained; a shear rate of zero or below needs acquisition context,Yes — review with the non-positive responses,value_qc_findings.csv
6,46 viscosity worksheets include shear rates below the paper's stated 0.01 s⁻¹,Verified,Measured range differs from the published method; points retained,No action in Phase 02,context_review.csv
7,46 of 46 modulus worksheets fixed at 1 Hz,Assumption,Consistent with the paper's amplitude sweeps; the linear viscoelastic region is not assigned,No,measurement_context_summary.csv
8,Image blanks recorded as unknown_blank: completeness_sam 943; contain_sps 922,Verified,"No blank is read as 'no', zero or a failed print",No,image_missingness.csv
9,Intensity: the paper supports a 0–100 % light-setting reading,Unresolved,"Partly clarified; the CSV unit and calibration are not confirmed, so no irradiance is calculated",Yes — confirm with the methods or authors,phase2_issue_decisions.csv


**Handoff to Phase 03.** The 26 tables in `data/metadata/schema_and_quality_control/` define the schema, its data dictionary and QC rules, and hold the source-preserving tables that Phase 03 standardises. Phase 03 re-verifies every table against the SHA-256 values in `schema_qc_run_manifest.json`.